# 13장 실습 ③ — 이상 탐지가 **안 되는** 경우

**Keras 3 판**

*"정상만 보고 배운 모델은 이상한 것을 잘 못 그린다"* 는 착상입니다.
널리 소개되는 용법인데, **이 경우에는 안 됩니다.**

실습 ④와 함께 보셔야 합니다.

## 13.0 준비

In [1]:
try:
    import dlbook
except ImportError:
    !pip install -q "dlbook @ git+https://github.com/dhrim/deep-learning-in-one-semester.git"
    import dlbook

In [2]:
import numpy as np
import matplotlib.pyplot as plt

import dlbook
from dlbook import data, metrics, plot

dlbook.set_seed(42)
plot.use_korean()
print(dlbook.versions())

{'python': '3.12.3', 'numpy': '2.1.3', 'keras': '3.15.1', 'tensorflow': '2.21.0', 'torch': '-', 'keras_backend': 'tensorflow'}


## 13.1 데이터 — 정답을 쓰지 않습니다

이 장에서 처음으로 `y` 를 쓰지 않습니다.

In [3]:
# MNIST. **정답(y)을 쓰지 않습니다.** 입력이 곧 정답입니다.
s = data.mnist()
x_train = s.x_train.reshape(len(s.x_train), -1)      # (N, 784)
x_test = s.x_test.reshape(len(s.x_test), -1)
print(f"학습 {x_train.shape}, 시험 {x_test.shape}")
print()
print("★ y_train 을 한 번도 쓰지 않습니다. 이것이 비지도 학습입니다.")

def show(rows, titles, n=8):
    """여러 줄의 28x28 영상을 나란히 그린다."""
    fig, axes = plt.subplots(len(rows), n, figsize=(1.15 * n, 1.25 * len(rows)))
    axes = np.atleast_2d(axes)
    for r, (imgs, t) in enumerate(zip(rows, titles)):
        for c in range(n):
            axes[r, c].imshow(imgs[c].reshape(28, 28), cmap="gray", vmin=0, vmax=1)
            axes[r, c].axis("off")
        axes[r, 0].set_ylabel(t)
        axes[r, 0].axis("on"); axes[r, 0].set_xticks([]); axes[r, 0].set_yticks([])
    plt.tight_layout(); plt.show()

학습 (54000, 784), 시험 (10000, 784)

★ y_train 을 한 번도 쓰지 않습니다. 이것이 비지도 학습입니다.


## 13.3 모델 정의 — 여기만 판마다 다릅니다

인코더로 줄이고 디코더로 되살립니다. **가운데가 좁은 것**이 전부입니다.

In [4]:
import keras
from keras import layers

def _build(latent, linear=False):
    """인코더와 디코더 — **이 함수만 판마다 다릅니다.**"""
    act = None if linear else "relu"
    out_act = None if linear else "sigmoid"
    encoder = keras.Sequential([
        layers.Input(shape=(784,)),
        layers.Dense(128, activation=act),
        layers.Dense(latent, activation=act),
    ], name="encoder")
    decoder = keras.Sequential([
        layers.Input(shape=(latent,)),
        layers.Dense(128, activation=act),
        layers.Dense(784, activation=out_act),
    ], name="decoder")
    return keras.Sequential([encoder, decoder]), encoder, decoder

def train_ae(latent, xa, xb, y_train=None, y_test=None,
             linear=False, seed=42, epochs=15):
    """(시험 복원 MSE, 모델) 을 돌려준다.

    y_train 을 주지 않으면 **입력이 곧 정답**이다. 주면 잡음 제거가 된다.
    """
    dlbook.set_seed(seed)
    m, enc, dec = _build(latent, linear)
    m.compile(optimizer=keras.optimizers.Adam(0.001), loss="mse")
    m.fit(xa, xa if y_train is None else y_train,
          epochs=dlbook.smoke.epochs(epochs), batch_size=256, verbose=0)
    target = xb if y_test is None else y_test
    rec = m.predict(xb, verbose=0)
    return float(np.mean((rec - target) ** 2)), m

def reconstruct(model, X):
    return model.predict(X, verbose=0)

def encode(model, X):
    return model.layers[0].predict(X, verbose=0)

## 13.1 이상 탐지 — 같은 데이터 안의 새 종류

MNIST에서 **0~8만** 학습하고, 시험에서 **9**를 찾아냅니다.

In [5]:
def recon_error(model, X):
    return np.mean((reconstruct(model, X) - X) ** 2, axis=1)

def auc(score, label):
    """ROC 곡선 아래 면적. 순위만 쓰므로 임계값을 안 정해도 된다."""
    order = np.argsort(score); r = np.empty(len(score))
    r[order] = np.arange(1, len(score) + 1)
    n1 = int(label.sum()); n0 = len(label) - n1
    return float((r[label == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))

In [6]:
# 같은 데이터 안의 새 숫자. 0~8만 배우고 9를 찾아낸다.
keep = s.y_train != 9

print(f"{'잠재':<8}{'정상 오차':>12}{'9의 오차':>12}{'AUC':>10}")
print("-" * 42)
for L in ([8] if dlbook.smoke.is_smoke() else [2, 8, 32]):
    _, m = train_ae(L, x_train[keep], x_test)
    e = recon_error(m, x_test)
    a = auc(e, (s.y_test == 9).astype(int))
    print(f"{L:<8}{e[s.y_test != 9].mean():>12.5f}"
          f"{e[s.y_test == 9].mean():>12.5f}{a:>10.3f}")
    dlbook.record(f"ch13_anomaly_digit9_auc_{L}", a)

print()
print("★ **거의 안 됩니다.** AUC가 0.5(찍기) 근처입니다.")
print("  0~8을 배우며 익힌 획과 곡선으로 9도 그대로 그려 버립니다.")
print("  모델이 배운 것은 '내가 본 아홉 종류'가 아니라 **'손으로 쓴 획'** 입니다.")

잠재             정상 오차       9의 오차       AUC
------------------------------------------


2            0.04617     0.04678     0.491
ch13_anomaly_digit9_auc_2 = 0.4912


8            0.02462     0.02840     0.599
ch13_anomaly_digit9_auc_8 = 0.5992


32           0.01033     0.01023     0.506
ch13_anomaly_digit9_auc_32 = 0.5056

★ **거의 안 됩니다.** AUC가 0.5(찍기) 근처입니다.
  0~8을 배우며 익힌 획과 곡선으로 9도 그대로 그려 버립니다.
  모델이 배운 것은 '내가 본 아홉 종류'가 아니라 **'손으로 쓴 획'** 입니다.


## 정리 — 그리고 다음 노트북으로

**AUC 0.5는 찍기입니다.** 이 방법이 여기서는 통하지 않습니다.

0~8을 배우며 익힌 것은 *"숫자 9의 생김새"* 가 아니라 **획, 곡선, 굵기**
같은 일반적인 부품입니다. 그 부품으로 9도 그대로 그릴 수 있습니다.

**→ 그런데 언제나 안 되는 것은 아닙니다. `ch13_anomaly_ood.ipynb` 로
가십시오.**

### 연습

1. 이상으로 둘 숫자를 0, 1, 8로 바꾸시오. **어느 숫자가 가장 잘
   잡힙니까. 왜입니까.**
2. 복원 오차 대신 **잠재 벡터까지의 최근접 거리**로 점수를 매기면
   나아집니까.